# PACT: bounded warm-start GPU check

Review and publish this code, then pin its full commit below. Select one BF16-capable GPU.
This notebook uses 12 frozen **training** tasks. The default execution stops after one
optimizer update (four examples); the complete recipe is nine updates across three adapters.
Set `EXECUTE = True` deliberately to load Qwen3-8B and train. This is an engineering check,
not a PACT effectiveness result. No final-test data is accessed.


In [ ]:
REPO_URL = "https://github.com/Soqoro/Pact.git"
GIT_REF = ""  # full reviewed and published commit SHA
RUN_ID = "qwen3-warmstart-001"
SCRATCH_ROOT = "/content/pact-scratch"
PERSISTENT_ROOT = "/content/drive/MyDrive/PACT"
MOUNT_DRIVE = True
PRIVATE_REPOSITORY = False
EXECUTE = False  # set True for the bounded GPU training check
RESUME = False
STOP_AFTER = 1  # new updates this invocation; None completes the remaining recipe
RESTORE_SNAPSHOT = ""  # exact verified snapshot path printed by the prior invocation
STORAGE_TIMEOUT_SECONDS = 120  # raise explicitly if healthy large copies need longer


Optional Drive mounting and code checkout. For private GitHub repositories, add `PACT_GITHUB_TOKEN` to Colab Secrets or enter it in the hidden prompt. The credential is passed to Git through a temporary askpass helper; it is never embedded in a URL, saved in Git configuration, or printed.


In [ ]:
from pathlib import Path
import getpass, os, re, subprocess, sys, tempfile
from urllib.parse import urlsplit

if not re.fullmatch(r"[0-9a-fA-F]{40}", GIT_REF):
    raise ValueError("Set GIT_REF to the full reviewed commit SHA.")
url = urlsplit(REPO_URL)
if url.scheme != "https" or not url.hostname or url.username or url.password or url.query or url.fragment:
    raise ValueError("Use a credential-free HTTPS repository URL.")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
CHECKOUT = Path(SCRATCH_ROOT) / "checkout"
CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
secret = None
if PRIVATE_REPOSITORY:
    try:
        from google.colab import userdata
        secret = userdata.get("PACT_GITHUB_TOKEN")
    except Exception:
        secret = getpass.getpass("GitHub token (hidden): ")

def git_checked(arguments, env):
    result = subprocess.run(["git", "-c", "credential.helper=", *arguments], env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Git checkout failed. Check URL, commit, and runtime credential; command output suppressed to protect credentials.")
    return result.stdout.strip()

try:
    with tempfile.TemporaryDirectory(prefix="pact-auth-") as auth_dir:
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        if secret:
            helper = Path(auth_dir) / "askpass.py"
            helper.write_text('#!/usr/bin/env python3\nimport os, sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["PACT_GIT_TOKEN"])\n')
            helper.chmod(0o700)
            env.update(GIT_ASKPASS=str(helper), PACT_GIT_TOKEN=secret)
        if not CHECKOUT.exists():
            git_checked(["clone", "--filter=blob:none", REPO_URL, str(CHECKOUT)], env)
        if git_checked(["-C", str(CHECKOUT), "status", "--porcelain"], env):
            raise RuntimeError("Checkout has local changes; preserve them before switching code.")
        origin = git_checked(["-C", str(CHECKOUT), "remote", "get-url", "origin"], env)
        if origin != REPO_URL:
            raise RuntimeError("Existing checkout has a different origin; choose a new scratch root.")
        git_checked(["-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", GIT_REF], env)
        git_checked(["-C", str(CHECKOUT), "checkout", "--detach", GIT_REF], env)
        actual = git_checked(["-C", str(CHECKOUT), "rev-parse", "HEAD"], env)
        if actual.lower() != GIT_REF.lower():
            raise RuntimeError("Resolved code does not match the requested commit.")
        print("Pinned code commit:", actual)
finally:
    secret = None
    if "env" in globals():
        env.pop("PACT_GIT_TOKEN", None)

os.chdir(CHECKOUT)
sys.path.insert(0, str(CHECKOUT / "src"))


Install the pinned libraries while preserving Colab's PyTorch build. Run the offline
CPU suite before using the GPU. GPU compatibility is checked by the training loader.


In [ ]:
from pact.colab import install_dependencies, cpu_checks
install_dependencies(CHECKOUT)
cpu_checks(CHECKOUT)


Prepare the fixed 12-task training manifest. Missing pinned source files are downloaded;
existing files are verified. Selection and overlap checks use the full training/validation
source pools; labels remain separate from prompts. A changed manifest fails the recipe check.


In [ ]:
from pact.cli import main
from pact.training.warmstart_config import load_warmstart_config, warmstart_plan

DATA_DIR = Path(SCRATCH_ROOT) / "training-data" / "engineering-12"
if not DATA_DIR.exists():
    status = main(["prepare-training-data", "--cache-dir", str(Path(SCRATCH_ROOT) / "cache" / "datasets"),
                   "--output-dir", str(DATA_DIR), "--items", "12", "--seed", "20260918", "--download"])
    if status:
        raise RuntimeError("Training data preparation failed; inspect the error above.")
CONFIG = CHECKOUT / "experiments" / "warmstart_engineering.json"
PLAN = warmstart_plan(load_warmstart_config(CONFIG), DATA_DIR)[3]
print(PLAN)


After a runtime reset, set `RESUME=True` and copy the **exact** printed snapshot path
into `RESTORE_SNAPSHOT`. Restore checks all file hashes and the checkpoint sequence in a
worker with a deadline. It refuses an existing local run directory and never falls back
to an older snapshot. Keep the same commit, recipe and compatible hardware/runtime.
Skip restore for an existing intact scratch run; set `RESTORE_SNAPSHOT=""` in that case.


In [ ]:
from pact.training.colab import restore_snapshot
RUN_DIR = Path(SCRATCH_ROOT) / "warmstart" / RUN_ID
if RESTORE_SNAPSHOT:
    if not RESUME:
        raise ValueError("Snapshot restore requires RESUME=True.")
    print(restore_snapshot(RESTORE_SNAPSHOT, RUN_DIR, timeout_seconds=STORAGE_TIMEOUT_SECONDS))


Run the first update, then return its review ZIP before expanding the check. A local ZIP
is created before Drive work. Full checkpoints are copied to verified immutable snapshots;
the small ZIP contains metadata/logs/hashes, **not weights**, and cannot resume training.
Large checkpoint copies may need more than 120 seconds. On timeout keep the runtime alive
and retry only the persistence cell below with an explicitly increased timeout.


In [ ]:
from pact.training.colab import execute
if not EXECUTE:
    print("Plan only. Set EXECUTE=True to run the bounded training check.")
else:
    HANDOFF = execute(checkout=CHECKOUT, data_dir=DATA_DIR, run_id=RUN_ID,
                      scratch=SCRATCH_ROOT, persistent=PERSISTENT_ROOT, resume=RESUME,
                      stop_after=STOP_AFTER, timeout_seconds=STORAGE_TIMEOUT_SECONDS)
    print("Bring back:", HANDOFF["persistent_bundle"] or HANDOFF["path"])
    print("SHA256:", HANDOFF["sha256"])
    print("Verified snapshot:", HANDOFF["persistent_snapshot"])
    if HANDOFF["exit_code"]:
        raise RuntimeError("Training or persistence failed; preserve scratch and inspect the review ZIP.")


Optional persistence-only retry, without loading a model or repeating an update.
Leave `RETRY_PERSISTENCE=False` during Run All. After a Drive timeout, set it True and
rerun only this cell; increase `STORAGE_TIMEOUT_SECONDS` if appropriate. Successful storage
does not turn a failed training invocation into a successful one.


In [ ]:
RETRY_PERSISTENCE = False
if RETRY_PERSISTENCE:
    from pact.training.colab import handoff
    from pact.util import read_json
    HANDOFF = handoff(root=RUN_DIR, bundles=Path(SCRATCH_ROOT) / "bundles",
                      persistent=PERSISTENT_ROOT, run_id=RUN_ID,
                      outcome=read_json(RUN_DIR / "invocation.json"),
                      timeout_seconds=STORAGE_TIMEOUT_SECONDS)
    print("Bring back:", HANDOFF["persistent_bundle"] or HANDOFF["path"])
    print("SHA256:", HANDOFF["sha256"])
    print("Verified snapshot:", HANDOFF["persistent_snapshot"])


Download the small review ZIP into local `results_import/`. Preserve the complete Drive
`warmstart/<RUN_ID>/` object store and snapshot index; copying just the snapshot folder
is insufficient. Do not reset until `persistent_copy_verified` is true. Following review,
use the same recipe/commit with `RESUME=True`, `STOP_AFTER=None` to complete the remaining
eight updates. GPU resume is unverified until that return bundle has been audited.
